# Using Dask to burst jobs onto many CPUs
May 10, 2026

# Notebook to process data in SMCE helio-public S3 bucket using Dask

<i>Note:  This document is maintained through the SMCE HelioCloud github at github.com/heliocloud-data.</i>

This is a simple example showing how to get a list of FITS files, run them through Dask workers to pull them from disk and examine the header keyword(s).  The focus is not FITS handling per se, but the mechanics of accessing S3-backed object storage and distributing I/O-bound workloads across workers. The goal is to understand how to access inexpensive object storage and on-demand extra CPUs rather than laptop storage and laptop compute.

You can use this notebook to test out various parameters you might feed to Dask; Consider the 'batch size', number of workers, number of cores per worker and memory per worker. Use the dashboard link to inspect how Dask is performing. Try both manual and automatic scaling strategies. See if you can get it to process 360 files in 20 sec or less!

## What is Dask?

Dask is the tool for accessing additional CPUs quickly for your calculation.  Dask 'spins up' a cluster of temporary CPUs, you then throw your job at them and it intelligently distributes the workload, then gathers the results.  Typically up to 100 of these temporary workers are allowed for a HelioCloud.  Our quick demo here uses 5 CPUs ('workers'), each with 2 cores.

Dask provides distributed task scheduling and parallel execution across multiple worker processes or nodes.

A Dask cluster consists of:
- A scheduler (task graph orchestration)
- Workers (execution processes with configurable cores and memory)

In this context, Dask is used to parallelize many independent S3 object fetches and computations. This pattern is effective when latency and transfer time dominate over per-task CPU cost.

#### Import libraries, initialize objects and bring up cluster options widget

Instantiate a cluster and connect a `Client`. The cluster widget exposes scaling controls and links to the scheduler dashboard.

In [ ]:
from dask_gateway import Gateway, GatewayCluster
gateway = Gateway()
options = gateway.cluster_options()

# We're setting some defaults here just for grins... 
# I like the pangeo/base-notebook image for the workers since it has almost every library you'd need on a worker
# In our environment, without setting these, the widget will default to the same image that the notebook itself is running, 
# as well as 2 cores and 4GB memory per worker

options.worker_cores=2
options.worker_memory=1

# This calls the widget
options  

#### Initialize the cluster and assign the client to the cluster, display the cluster widget

In [ ]:
cluster = gateway.new_cluster(options)
client = cluster.get_client()
n_workers = 5
cluster.scale(n_workers)
client.wait_for_workers(n_workers)
cluster

#### Don't forget to scale your cluster to add workers, either manually or using adaptive scaling
#### Also, see that link in the cluster widget that says "Dashboard:"?  Click it to get to the Dask cluster dashboard
Remember, you can copy that URL into the Dask panel on the left

#### Now let's do some work.  Since we've assigned a client to the cluster object, any call to a dask object will automatically get sent to the cluster workers 

In [ ]:
from dask.distributed import Client

client = Client(cluster)
client

In [ ]:
%%time

import dask.array as da
a = da.random.normal(size=(30000,30000), chunks=(1000, 1000))
out = a.mean().compute()
print(out)


#### When you're finished, shut down the cluster to release the resources (it'll happen automatically after a set idle time)

In [ ]:
client.close()
cluster.shutdown()

A few troubleshooting functions

In [ ]:
gateway.get_versions()

In [ ]:
gateway.list_clusters()

### Scaling and Monitoring

Explicitly scale the cluster (fixed worker count) or enable adaptive scaling.

Use the Dask dashboard to inspect:
- Task graph execution
- Worker utilization
- Network transfer
- Memory pressure

When working against S3, network and object fetch latency are often the primary constraints.

## With Actual Data

### Example: Processing Public SDO AIA Data

We now perform a representative workflow:

1. Enumerate SDO AIA FITS objects in a public S3 bucket.
2. Launch a Dask cluster.
3. Distribute object fetch + header inspection across workers.
4. Aggregate results.

The key objective is to illustrate how object storage access patterns interact with distributed task scheduling.

Summarizing, here is a sample scientific task.  We will fetch SDO AIA files stored in the NASA public (ODR) S3 bucket, spin up a Dask cluster of 10 workers, then send 1 day's worth of data (360 files) to compute the average irradiance, and plot it.

In [ ]:
# setup to access S3 transparently
import boto3
from botocore import UNSIGNED
from botocore.config import Config

# necessary imports for Dask
import dask
from dask.distributed import Client

# imports for handling scientific files and doing basic searching
import io
import re
import logging
import cloudcatalog
from astropy.io import fits

## 0. Fetch cloud filelists for AIA

Here we grab a list of one day of AIA 0094A files.

In [ ]:
fr=cloudcatalog.CloudCatalog("s3://gov-nasa-hdrl-data1/")
frID = "aia_0094"
myjson = fr.get_entry(frID)
start, stop = myjson['start'], myjson['stop']
file_registry_aia = fr.request_cloud_catalog(frID, start_date=start, stop_date=stop, overwrite=False)
filelist = file_registry_aia['datakey'].to_list()
s3_files = filelist[0:360] # small test set to test
print(f"Example file URI: {s3_files[0]}")

## 1. Configuration

Define tunable parameters:

- Worker count
- Cores per worker
- Memory per worker
- Batch size (objects per task)

These parameters affect concurrency, memory footprint, and S3 request rate.

Adjust configuration values to explore trade-offs between:

- High concurrency (many small tasks)
- Larger batch sizes (fewer S3 round-trips)
- Memory constraints per worker
- Network saturation

(Feel free to play with the following values to optimize the performance.)

In [ ]:
# number of workers to use, for automatic scaling, our max number
n_workers = 10

# memory per worker (in Gb)
w_memory = 2

# cores per worker
w_cores = 2

# number of files to test against (360 max)
n_files = 100

# Number of files we release to be worked on by all workers at a time
# the higher the number the more files being processed concurrently, but also
# the greater the memory consumed. 
batch_size = 50

## 2. Initialize the cluster and assign the client to the cluster, display the cluster widget

### Initialize Cluster

Create and configure the Dask cluster, then attach a `Client`. Confirm connectivity via the cluster widget.

In [ ]:
from dask_gateway import Gateway, GatewayCluster
gateway = Gateway()
options = gateway.cluster_options()

# We're setting some defaults here just for grins... 
# I like the pangeo/base-notebook image for the workers since it has almost every library you'd need on a worker
# In our environment, without setting these, the widget will default to the same image that the notebook itself is running, 
# as well as 2 cores and 4GB memory per worker

options.worker_cores=w_cores
options.worker_memory=w_memory
# options

In [ ]:
cluster = gateway.new_cluster(options)
client = cluster.get_client()

# use Manual (if False, then uses Automatic scaling)
use_manual_scaling = True

if use_manual_scaling:
    # manual scaling (n_workers defined above)
    cluster.scale(n_workers)
else:
    # Adaptively scale between 1 and n_workers (the max)
    cluster.adapt(minimum=1, maximum=n_workers)
client.wait_for_workers(n_workers)
# uncomment this if you want to use the GUI
#cluster

In [ ]:
# create client, show url we can go to to monitor progress
client = Client(cluster)
client

## 3. Define some routines we will use for doing work with Dask

### Define Worker Routines

Define functions that:

- Retrieve objects from S3
- Parse FITS headers
- Extract relevant metadata

Each invocation results in one or more remote object fetches. Efficient design minimizes redundant reads and maximizes sequential access within a task.

In [ ]:
import time
import astropy.io.fits
import s3fs
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
%matplotlib inline 
# Optional, have Matplotlib create vector (svg) instead of raster (png) images
%config InlineBackend.figure_formats = ['svg'] 

def DO_SCIENCE(mydata):
    # you can put better science here
    iirad = mydata.mean()
    return iirad

# these are variable helpful handler functions
def s3url_to_bucketkey(s3url: str): # -> Tuple[str, str]:
    """
    Extracts the S3 bucket name and file key from an S3 URL.
    e.g. s3://mybucket/mykeypart1/mykeypart2/fname.fits -> mybucket, mykeypart1/mykeypart2/fname.fits
    """
    name2 = re.sub(r"s3://","",s3url)
    s = name2.split("/",1)
    return s[0], s[1]

def process_fits_s3(s3key:str): # -> Tuple[str, float]:
    """ For a single FITS file, read it from S3, grab the header and
        data, then do the DO_SCIENCE() call of choice
    """
    sess = boto3.session.Session() # do this each open to avoid thread problem 'credential_provider'
    s3c = sess.client("s3")
    mybucket,mykey = s3url_to_bucketkey(s3key)
    try:
        fobj = s3c.get_object(Bucket=mybucket,Key=mykey)
        rawdata = fobj['Body'].read()
        bdata = io.BytesIO(rawdata)
        hdul = astropy.io.fits.open(bdata,memmap=False)        
        date = hdul[1].header['T_OBS']
        irrad = DO_SCIENCE(hdul[1].data)
        print(date,irrad)
    except:
        print("Error fetching ",s3key)
        date, irrad = None, None
        
    return date, irrad

def work_on_data (client:dask.distributed.client.Client, files:list=[])->int:
    """ 
    Main routine which Dask will use to 'do work'. Each worker will run this.
    """
    # simple version step 1, do it
    mean_irrad = client.map(process_fits_s3, s3_files)

    # trigger distributed task, marshall result back to local memory
    all_data = client.gather(mean_irrad)

    # return the primary header back for analysis
    return all_data

def plot_lightcurve(results):
    # Plot the Dask results aka the light curve
    if len(results) > 0:
        dates, values = zip(*results)
        plt.plot(dates, values) #, marker='o', linestyle='-')
        plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%d-%m-%Y'))
        plt.gca().xaxis.set_major_locator(mdates.AutoDateLocator())
        plt.xticks(rotation=90)
        plt.show()

## 4. Executing Work

### Do the cloud processing, using Dask to 'burst' into other VMs

Using our gathered list of FITS files, chunk it out in batches and provide file list chunks to the workers.

Once a `Client` is attached, Dask collections and delayed functions are executed on workers. Each worker independently retrieves required S3 objects via the S3 API.

### Distributed Processing

Partition the object list into batches and submit them to Dask workers.

Each worker:
1. Issues S3 GET requests for its assigned objects.
2. Streams object content over the network.
3. Performs local computation.
4. Returns aggregated results to the scheduler.

This “burst” pattern scales read concurrency but remains bounded by S3 request rate, object size, and network bandwidth. Understanding these constraints is essential when treating S3 as a scientific data backend.

In [ ]:
%%time
if n_files > len(s3_files):
    n_files = len(s3_files)
    
def chunks(lst, n):
    """ program to divide our file list into chunks for each worker """
    n = max(1, n)
    return (lst[i:i+n] for i in range(0, len(lst), n))

client.wait_for_workers(n_workers)

print (f"workers: {n_workers}, cores/worker:{w_cores}, mem/worker: {w_memory}")
for files_to_process in chunks(s3_files[:n_files], batch_size):
    returns = work_on_data(client, files_to_process)
    print (f"client:%s Finished %s files" % (client,len(returns)))

plot_lightcurve(returns)

In [ ]:
yn = input("Done with cluster, able to shut it down? y/n: ")
if yn.startswith('y'):
    client.close()
    cluster.shutdown()

## 5. Better Dask handling

The above routines use Dask fundamentals. However, better handling of the cluster setup, robust send/gather to dask itself, and cleaner cluster cleanup are recommended.  Here are several drop-in routines that are more fault tolerant.  Use these for your own code!


In [ ]:
from dask.distributed import Client, as_completed, wait
from dask_gateway import Gateway, GatewayCluster
from distributed.client import CancelledError  # same exception type raised by Future.result()
import logging
logging.getLogger("distributed.client").setLevel(logging.WARNING)

def setup_cluster(*, n_workers=4, cores=4, memory_gib=2, wait_workers_timeout_s=180):
    gateway = Gateway()
    options = gateway.cluster_options()
    options.worker_cores = cores
    options.worker_memory = float(memory_gib)

    cluster = gateway.new_cluster(options)
    cluster.scale(n_workers)
    client = cluster.get_client()
    client.wait_for_workers(n_workers, timeout=wait_workers_timeout_s)
    return gateway, cluster, client

def robust_gather(client, futures, first_results_timeout_s=600):
    futures = list(futures)
    ac = as_completed(futures)
    first = next(ac, None)
    if first is None:
        return [], []

    results = []
    errors = []
    try:
        results.append(first.result(timeout=first_results_timeout_s))
    except CancelledError as e:
        errors.append(("scheduler-connection-lost", repr(e)))
        return results, errors   # <-- MINIMAL FIX (was: raise)

    for fut in ac:
        try:
            results.append(fut.result())
        except CancelledError as e:
            errors.append(("scheduler-connection-lost", repr(e)))
            return results, errors   # <-- MINIMAL FIX (was: raise)
        except Exception as e:
            errors.append((fut.key, repr(e)))

    return results, errors

def clean_shutdown(client, cluster):
    if client is not None:
        try:
            client.close()
        except Exception:
            pass
    if cluster is not None:
        try:
            cluster.shutdown()
        except Exception:
            pass
            
def setup_cluster(*, n_workers=4, cores=4, memory_gib=2, wait_workers_timeout_s=180):
    gateway = Gateway()
    options = gateway.cluster_options()
    options.worker_cores = cores
    options.worker_memory = float(memory_gib)

    cluster = gateway.new_cluster(options)
    cluster.scale(n_workers)
    client = cluster.get_client()
    client.wait_for_workers(n_workers, timeout=wait_workers_timeout_s)
    return gateway, cluster, client

def robust_gather(client, futures, first_results_timeout_s=600):
    futures = list(futures)
    ac = as_completed(futures)
    first = next(ac, None)
    if first is None:
        return [], []

    results = []
    errors = []
    try:
        results.append(first.result(timeout=first_results_timeout_s))
    except CancelledError as e:
        errors.append(("scheduler-connection-lost", repr(e)))
        return results, errors   # <-- MINIMAL FIX (was: raise)

    for fut in ac:
        try:
            results.append(fut.result())
        except CancelledError as e:
            errors.append(("scheduler-connection-lost", repr(e)))
            return results, errors   # <-- MINIMAL FIX (was: raise)
        except Exception as e:
            errors.append((fut.key, repr(e)))

    return results, errors

def clean_shutdown(client, cluster): #, gateway):
    if client is not None:
        try:
            client.close()
        except Exception:
            pass
    if cluster is not None:
        try:
            cluster.shutdown()
        except Exception:
            pass

In [ ]:
# Same run as before, but using our better helper routines.

n_workers = 4
n_cores = 4
memory_gib = 2
now = time.time()
# now we initialize the dask cluster.
try:
    gateway, cluster, client = setup_cluster(n_workers=n_workers,cores=n_cores,memory_gib=memory_gib,wait_workers_timeout_s=180)
    futures = client.map(process_fits_s3, s3_files, retries=2)
    results, errors = robust_gather(client,futures,first_results_timeout_s=600)
    print("Cluster setup and run took", (time.time() - now) / 60.0, "minutes on", len(results),"files (",len(errors),"errors) out of ",len(s3_files),"input files")
except Exception as e:
    print("Error, cluster not set up or timed out.", type(e).__name__, e)
finally:
    clean_shutdown(client,cluster)
plot_lightcurve(results)